# RAG (Retrieve Augumented Generation) Pipeline

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from {PDF_PATH}")
print(f"First page preview (first 500 chars")
print(pages[0].page_content[:500])

/tmp/ipykernel_1383/2638228594.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from telecom_guide.pdf
First page preview (first 500 chars
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [3]:
print(pages[2].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 


## Split the text

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, # ~150 words per chunk
    chunk_overlap=200, # Overlap keeps context at boundaries
    length_function=len,
    separators=["\n\n", "\n", " ", "", "."] # tries paragraph -> line -> sentence -> word
)


chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunks")

Split into 44 chunks


In [5]:
print(f"First chunk preview: {chunks[0].page_content}")
print("\n")
print(f"Second chunk preview: {chunks[1].page_content}")

First chunk preview: Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


Second chunk preview: Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app


## Do embedding and store them in ChromaDB

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store created with {vector_store._collection.count()} vectors")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1403.21it/s]


Vector store created with 44 vectors


## Retriever

In [9]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VOLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 0 ---
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navig

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
the device may not support VoLTE or the profile has not been pushed to the SIM. Agents can push the VoLTE
profile remotely via the subscriber management system.
VoWiFi (Wi-Fi Calling): VoWiFi extends IMS calling over any Wi-Fi network, including home broadband and public
hotspots. This is especially



Sure! Here's the Markdown laid out **horizontally** to closely match the flow of the diagram.

````markdown
# Retrieval

| User Query | Query Embedding | Vector DB | Retrieved Chunks | Prompt to LLM | LLM | Answer |
|------------|-----------------|-----------|------------------|---------------|-----|--------|
| **How much money does AtliQ Technologies contribute to an employee's retirement fund?** | ```text [ 0.18, -0.97, 0.60, ... ] ``` | 🗄️ **Vector DB** | **Chunk 1**<br><br>Employee's provident fund (EPF).<br>All eligible employees...<br><br>**Chunk 2**<br><br>AtliQ's voluntary retirement.<br>Retirement policy... | **Question (Q):**<br>How much money does AtliQ...<br><br>**Answer based on the following:**<br>- Chunk 1<br>- Chunk 2 | 🤖 LLM | ✅ Answer |

````

Or, if you want it to look even closer to the image using arrows:

````markdown
# Retrieval

```text
┌───────────────────────────────────────────────────────────────┐
│ User Query                                                    │
│                                                               │
│ How much money does AtliQ Technologies contribute to an       │
│ employee's retirement fund?                                   │
└───────────────────────────────────────────────────────────────┘
                            │
                            ▼
                  Query Embedding
                [0.18, -0.97, 0.60, ...]

                            │
                            ▼
                      ┌────────────┐
                      │ Vector DB  │
                      └────────────┘
                            │
            ┌───────────────┴───────────────┐
            ▼                               ▼

      Embedding 1                     Embedding 2
   [0.2, -1.4, 0.8...]             [1.3, 0.7, 0.4...]

            │                               │
            ▼                               ▼

    ┌─────────────────────┐        ┌─────────────────────┐
    │ Chunk 1             │        │ Chunk 2             │
    │ Employee's EPF      │        │ AtliQ's voluntary   │
    │ All eligible        │        │ retirement policy   │
    │ employees...        │        │ ...                 │
    └─────────────────────┘        └─────────────────────┘
             \                           /
              \                         /
               \                       /
                ▼                     ▼

      ┌───────────────────────────────────────────┐
      │ Question (Q):                             │
      │ How much money does AtliQ...?             │
      │                                           │
      │ Answer based on:                          │
      │   • Chunk 1                               │
      │   • Chunk 2                               │
      └───────────────────────────────────────────┘
                            │
                            ▼
                      ┌──────────┐
                      │   LLM    │
                      └──────────┘
                            │
                            ▼
                        ✅ Answer
````

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: Join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])


SYSTEM_PROMPT = """
    You are a helpful telecom assistant.
    Answer the question using ONLY the context provided below.
    If the context does not contain enough information, say so clearly.

    Context:
    {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", "{question}") # * Dynamically Supply the question at runtime to langchain
])



# --- LLM Via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain Assembled")

RAG Chain Assembled


In [12]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question} \n")
print("A: ", chain.invoke(question))

Q: How does international roaming work and what charges should I expect? 

A:  International roaming allows your device to connect to partner networks in foreign countries when you're outside your home network's coverage. Here's how it works and what charges to expect:

### **How It Works**  
1. **Network Connection**: When traveling abroad, your device automatically connects to a partner network in the visited country.  
2. **Authentication**: The foreign network verifies your subscription with your home network (via protocols like SS7 or Diameter). Your home network then authorizes services (calls, data, SMS).  

### **Charges**  
- **Zone-Based Pricing**:  
  - **Zone A** (e.g., Australia, New Zealand): Low per-MB and per-minute rates.  
  - **Zone B** (e.g., Japan, Singapore): Moderate rates.  
  - **Zone C** (Rest of World): Highest charges for data, calls, and SMS.  

- **Avoid Bill Shock**: Always purchase a **roaming bundle** before departure. Without a bundle, charges can be s